# Credit Risk Assessment: Phase 1 - Data Preprocessing
## Predictive Analytics-Based Decision Support Framework for Credit Risk Assessment in the Banking Sector

### Objective
Predict loan default risk using machine learning and implement a comprehensive data preprocessing pipeline.

**Target Variable:** Default Classification
- 1 = Default / Charged Off
- 0 = Fully Paid

### Pipeline Overview
1. ✅ Load and Explore Dataset
2. ✅ Data Cleaning and Missing Values
3. ✅ Binary Target Variable Creation
4. ✅ Feature Engineering and Selection
5. ✅ Categorical Variable Encoding
6. ✅ Feature Scaling and Normalization
7. ✅ Handling Class Imbalance with SMOTE

**Dataset:** LC_loans_granting_model_dataset.csv (1,347,681 records)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# Set visualization style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

print("✓ All libraries imported successfully")
print("✓ Environment configured")

## Configuration and Constants

In [ ]:
# Configuration
DATA_PATH = '../data/raw/LC_loans_granting_model_dataset.csv'
OUTPUT_DIR = '../data/processed/'
RANDOM_STATE = 42

# Feature categories
NUMERIC_FEATURES = ['revenue', 'dti_n', 'loan_amnt', 'fico_n', 'experience_c']
CATEGORICAL_FEATURES = ['emp_length', 'purpose', 'home_ownership_n', 'addr_state']
TARGET_COLUMN = 'Default'

# Columns to drop
DROP_COLUMNS = ['id', 'zip_code', 'title', 'desc', 'issue_d']

# Parameters
MISSING_VALUE_THRESHOLD = 0.5  # Drop columns with >50% missing
SMOTE_K_NEIGHBORS = 5
TEST_SIZE = 0.2

print("✓ Configuration loaded")

---

## Section 1: Load and Explore the Dataset

In [ ]:
# Step 1: Load Dataset
print("="*80)
print("STEP 1: LOADING DATASET")
print("="*80)

df = pd.read_csv(DATA_PATH, low_memory=False)
print(f"\n✓ Dataset loaded successfully!")
print(f"  • Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"  • Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

In [ ]:
# Step 2: Display Dataset Structure
print("\n" + "="*80)
print("STEP 2: DATASET STRUCTURE AND SUMMARY")
print("="*80)

print("\n📋 COLUMNS:")
print(df.columns.tolist())

print("\n📊 DATA TYPES:")
print(df.dtypes)

print("\n📈 STATISTICAL SUMMARY:")
print(df.describe())

In [ ]:
print("\n📋 FIRST FEW ROWS:")
print(df.head())

print("\n📌 DATASET INFO:")
print(df.info())

---

## Section 2: Data Cleaning and Missing Values

In [ ]:
# Step 3: Handle Missing Values
print("\n" + "="*80)
print("STEP 3: DATA CLEANING - MISSING VALUES")
print("="*80)

print("\n📊 MISSING VALUES BEFORE CLEANING:")
missing_info = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({'Missing_Count': missing_info, 'Percentage': missing_pct})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Percentage', ascending=False)
print(missing_df)

print(f"\n  Total missing values: {df.isnull().sum().sum():,}")

# Create a copy for processing
df_clean = df.copy()

# Identify columns with high missing values
high_missing_cols = missing_df[missing_df['Percentage'] > MISSING_VALUE_THRESHOLD].index.tolist()

print(f"\n🔴 DROPPING COLUMNS WITH >{MISSING_VALUE_THRESHOLD*100}% MISSING:")
for col in high_missing_cols:
    pct = missing_df.loc[col, 'Percentage']
    print(f"  • {col}: {pct:.2f}% missing")
    
df_clean = df_clean.drop(columns=high_missing_cols)

print("\n🟢 FILLING REMAINING MISSING VALUES:")

# Fill numeric columns with median
numeric_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if df_clean[col].isnull().sum() > 0:
        median_val = df_clean[col].median()
        df_clean[col].fillna(median_val, inplace=True)
        print(f"  • {col}: Filled with median ({median_val:.2f})")

# Fill categorical columns with mode
categorical_cols = df_clean.select_dtypes(include=['object']).columns
for col in categorical_cols:
    if df_clean[col].isnull().sum() > 0:
        mode_val = df_clean[col].mode()[0]
        df_clean[col].fillna(mode_val, inplace=True)
        print(f"  • {col}: Filled with mode ({mode_val})")

print(f"\n✓ Missing values after cleaning: {df_clean.isnull().sum().sum()}")
print(f"  Dataset shape: {df_clean.shape}")

---

## Section 3: Binary Target Variable Creation

In [ ]:
# Step 4: Convert Target Variable to Binary
print("\n" + "="*80)
print("STEP 4: BINARY TARGET VARIABLE CONVERSION")
print("="*80)

print(f"\n📊 ORIGINAL TARGET DISTRIBUTION:")
print(f"  Unique values: {df_clean[TARGET_COLUMN].unique()}")
print(f"  Value counts:")
print(df_clean[TARGET_COLUMN].value_counts())

# Convert to binary: 1 = Default/Charged Off, 0 = Fully Paid
# Target is already numeric (0, 1), ensure proper format
df_clean[TARGET_COLUMN] = df_clean[TARGET_COLUMN].astype(int)

print(f"\n🎯 BINARY TARGET AFTER CONVERSION:")
print(f"  • Class 0 (Fully Paid): {(df_clean[TARGET_COLUMN] == 0).sum():,} samples ({(df_clean[TARGET_COLUMN] == 0).sum()/len(df_clean)*100:.2f}%)")
print(f"  • Class 1 (Default): {(df_clean[TARGET_COLUMN] == 1).sum():,} samples ({(df_clean[TARGET_COLUMN] == 1).sum()/len(df_clean)*100:.2f}%)")
print(f"  • Imbalance ratio: {(df_clean[TARGET_COLUMN] == 1).sum() / (df_clean[TARGET_COLUMN] == 0).sum():.4f}")

# Visualization
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = df_clean[TARGET_COLUMN].value_counts()
labels = ['Fully Paid (0)', 'Default (1)']
colors = ['#2ecc71', '#e74c3c']
ax[0].bar(labels, counts.values, color=colors, alpha=0.7, edgecolor='black')
ax[0].set_ylabel('Count', fontsize=12)
ax[0].set_title('Target Variable Distribution', fontsize=13, fontweight='bold')
ax[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(counts.values):
    ax[0].text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontweight='bold')

# Pie chart
ax[1].pie(counts.values, labels=labels, autopct='%1.2f%%', colors=colors, startangle=90)
ax[1].set_title('Target Variable Proportion', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Binary target variable created successfully")

---

## Section 4: Feature Engineering and Selection

In [ ]:
# Step 5: Remove Irrelevant Columns / Feature Selection
print("\n" + "="*80)
print("STEP 5: FEATURE ENGINEERING AND SELECTION")
print("="*80)

print(f"\n  Columns before feature selection: {df_clean.shape[1]}")
print(f"  {df_clean.columns.tolist()}")

# Remove irrelevant columns
cols_to_drop = [col for col in DROP_COLUMNS if col in df_clean.columns]
print(f"\n🔴 REMOVING IRRELEVANT COLUMNS:")
for col in cols_to_drop:
    print(f"  • {col}")
    
df_clean = df_clean.drop(columns=cols_to_drop)

print(f"\n✓ Columns after feature selection: {df_clean.shape[1]}")
print(f"  {df_clean.columns.tolist()}")

# Correlation analysis with target
print(f"\n📊 CORRELATION WITH TARGET VARIABLE:")
correlations = df_clean.corr()[TARGET_COLUMN].sort_values(ascending=False)
print(correlations)

# Visualization
fig, ax = plt.subplots(figsize=(10, 6))
correlations[correlations.index != TARGET_COLUMN].plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel('Correlation Coefficient', fontsize=12)
ax.set_title(f'Feature Correlation with {TARGET_COLUMN}', fontsize=13, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.8)
plt.tight_layout()
plt.show()

print(f"\n✓ Feature selection complete")

---

## Section 5: Categorical Variable Encoding

In [ ]:
# Step 6: Encode Categorical Variables
print("\n" + "="*80)
print("STEP 6: CATEGORICAL VARIABLE ENCODING")
print("="*80)

# Identify categorical columns
cat_cols_available = [col for col in CATEGORICAL_FEATURES if col in df_clean.columns]
print(f"\n📊 CATEGORICAL VARIABLES TO ENCODE:")
for col in cat_cols_available:
    unique_count = df_clean[col].nunique()
    print(f"  • {col}: {unique_count} unique values")

# Label Encoding
label_encoders = {}
print(f"\n🟢 APPLYING LABEL ENCODING:")

for col in cat_cols_available:
    le = LabelEncoder()
    df_clean[col] = le.fit_transform(df_clean[col].astype(str))
    label_encoders[col] = le
    print(f"  • {col}: {len(le.classes_)} classes encoded")
    print(f"    Classes: {list(le.classes_)}")

print(f"\n✓ Categorical encoding complete")
print(f"  Dataset shape: {df_clean.shape}")
print(f"\n✓ All features are now numeric and ready for scaling")

---

## Section 6: Feature Scaling and Normalization

In [ ]:
# Step 7: Feature Scaling
print("\n" + "="*80)
print("STEP 7: FEATURE SCALING (STANDARDIZATION)")
print("="*80)

# Separate features and target
X = df_clean.drop(columns=[TARGET_COLUMN])
y = df_clean[TARGET_COLUMN]

# Get numeric columns
numeric_cols_to_scale = [col for col in X.columns if X[col].dtype in [np.number]]
print(f"\n📊 NUMERIC FEATURES TO SCALE: {numeric_cols_to_scale}")

print(f"\n📈 BEFORE SCALING:")
print(f"  Mean: {X[numeric_cols_to_scale].mean().mean():.6f}")
print(f"  Std Dev: {X[numeric_cols_to_scale].std().mean():.6f}")
print(f"  Min: {X[numeric_cols_to_scale].min().min():.6f}")
print(f"  Max: {X[numeric_cols_to_scale].max().max():.6f}")

# Apply StandardScaler
scaler = StandardScaler()
X[numeric_cols_to_scale] = scaler.fit_transform(X[numeric_cols_to_scale])

print(f"\n🟢 USING STANDARDSCALER (Zero Mean, Unit Variance)")
print(f"\n📊 AFTER SCALING:")
print(f"  Mean: {X[numeric_cols_to_scale].mean().mean():.6f}")
print(f"  Std Dev: {X[numeric_cols_to_scale].std().mean():.6f}")
print(f"  Min: {X[numeric_cols_to_scale].min().min():.6f}")
print(f"  Max: {X[numeric_cols_to_scale].max().max():.6f}")

# Visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Distribution of features before scaling (using original df)
X_before = df_clean[numeric_cols_to_scale].copy()
axes[0, 0].hist(X_before['revenue'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Revenue Distribution (Before Scaling)', fontsize=11, fontweight='bold')
axes[0, 0].set_xlabel('Value')

axes[0, 1].hist(X_before['fico_n'], bins=50, color='steelblue', edgecolor='black', alpha=0.7)
axes[0, 1].set_title('FICO Score Distribution (Before Scaling)', fontsize=11, fontweight='bold')
axes[0, 1].set_xlabel('Value')

# Distribution after scaling
axes[1, 0].hist(X['revenue'], bins=50, color='green', edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Revenue Distribution (After Scaling)', fontsize=11, fontweight='bold')
axes[1, 0].set_xlabel('Scaled Value')

axes[1, 1].hist(X['fico_n'], bins=50, color='green', edgecolor='black', alpha=0.7)
axes[1, 1].set_title('FICO Score Distribution (After Scaling)', fontsize=11, fontweight='bold')
axes[1, 1].set_xlabel('Scaled Value')

plt.tight_layout()
plt.show()

print(f"\n✓ Feature scaling complete")
print(f"  All numeric features are now standardized")

---

## Section 7: Handling Class Imbalance with SMOTE

In [ ]:
# Step 8: Handle Class Imbalance with SMOTE
print("\n" + "="*80)
print("STEP 8: HANDLING CLASS IMBALANCE WITH SMOTE")
print("="*80)

print(f"\n📊 BEFORE SMOTE:")
print(f"  • Class 0 (Fully Paid): {(y == 0).sum():,} samples ({(y == 0).sum()/len(y)*100:.2f}%)")
print(f"  • Class 1 (Default): {(y == 1).sum():,} samples ({(y == 1).sum()/len(y)*100:.2f}%)")
print(f"  • Imbalance ratio: {(y == 1).sum() / (y == 0).sum():.4f}")
print(f"  • Total samples: {len(y):,}")

# Visualize class distribution before SMOTE
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Before SMOTE
counts_before = y.value_counts()
labels = ['Fully Paid (0)', 'Default (1)']
colors = ['#2ecc71', '#e74c3c']
axes[0].bar(labels, counts_before.values, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Class Distribution BEFORE SMOTE', fontsize=13, fontweight='bold')
axes[0].grid(axis='y', alpha=0.3)
for i, v in enumerate(counts_before.values):
    axes[0].text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontweight='bold')

# Apply SMOTE
print(f"\n🟢 APPLYING SMOTE...")
smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=SMOTE_K_NEIGHBORS)
X_smote, y_smote = smote.fit_resample(X, y)

print(f"\n📊 AFTER SMOTE:")
print(f"  • Class 0 (Fully Paid): {(y_smote == 0).sum():,} samples ({(y_smote == 0).sum()/len(y_smote)*100:.2f}%)")
print(f"  • Class 1 (Default): {(y_smote == 1).sum():,} samples ({(y_smote == 1).sum()/len(y_smote)*100:.2f}%)")
print(f"  • Imbalance ratio: {(y_smote == 1).sum() / (y_smote == 0).sum():.4f}")
print(f"  • Total samples: {len(y_smote):,}")
print(f"  • Synthetic samples generated: {len(y_smote) - len(y):,}")

# After SMOTE
counts_after = pd.Series(y_smote).value_counts()
axes[1].bar(labels, counts_after.values, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Class Distribution AFTER SMOTE', fontsize=13, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
for i, v in enumerate(counts_after.values):
    axes[1].text(i, v, f'{v:,.0f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print(f"\n✓ SMOTE applied successfully")
print(f"  Classes are now perfectly balanced for training")

---

## Final Step: Train-Test Split and Saving

In [ ]:
# Train-Test Split
print("\n" + "="*80)
print("TRAIN-TEST SPLIT")
print("="*80)

X_train, X_test, y_train, y_test = train_test_split(
    X_smote, y_smote,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y_smote
)

print(f"\n📊 DATA SPLIT SUMMARY:")
print(f"  • Training set: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X_smote)*100:.1f}%)")
print(f"  • Test set: {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X_smote)*100:.1f}%)")
print(f"  • Features: {X_train.shape[1]}")

print(f"\n  Training set class distribution:")
print(f"    • Class 0: {(y_train == 0).sum():,}")
print(f"    • Class 1: {(y_train == 1).sum():,}")

print(f"\n  Test set class distribution:")
print(f"    • Class 0: {(y_test == 0).sum():,}")
print(f"    • Class 1: {(y_test == 1).sum():,}")

# Prepare data for saving
balanced_df = pd.DataFrame(X_smote, columns=X.columns)
balanced_df[TARGET_COLUMN] = y_smote

train_df = X_train.copy()
train_df[TARGET_COLUMN] = y_train

test_df = X_test.copy()
test_df[TARGET_COLUMN] = y_test

print(f"\n✓ Train-test split completed")

In [ ]:
# Save Processed Data
print("\n" + "="*80)
print("SAVING PROCESSED DATA")
print("="*80)

# Create output directory if needed
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save files
balanced_df.to_csv(f'{OUTPUT_DIR}preprocessed_data.csv', index=False)
train_df.to_csv(f'{OUTPUT_DIR}train_data.csv', index=False)
test_df.to_csv(f'{OUTPUT_DIR}test_data.csv', index=False)

print(f"\n✓ Files saved successfully:")
print(f"  • {OUTPUT_DIR}preprocessed_data.csv ({balanced_df.shape[0]:,} rows, {balanced_df.shape[1]} cols)")
print(f"  • {OUTPUT_DIR}train_data.csv ({train_df.shape[0]:,} rows, {train_df.shape[1]} cols)")
print(f"  • {OUTPUT_DIR}test_data.csv ({test_df.shape[0]:,} rows, {test_df.shape[1]} cols)")

---

## Phase 1 Completion Summary

In [ ]:
print("\n" + "█"*80)
print("█" + " "*78 + "█")
print("█" + "PHASE 1: DATA PREPROCESSING COMPLETED SUCCESSFULLY!".center(78) + "█")
print("█" + " "*78 + "█")
print("█"*80)

print(f"""
╔════════════════════════════════════════════════════════════════════════════╗
║                         PREPROCESSING SUMMARY                             ║
╚════════════════════════════════════════════════════════════════════════════╝

📊 ORIGINAL DATASET:
   • Shape: {df.shape[0]:,} rows × {df.shape[1]} columns
   • Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB

📊 PROCESSED DATASET:
   • Shape: {balanced_df.shape[0]:,} rows × {balanced_df.shape[1]} columns
   • Features: {list(balanced_df.drop(TARGET_COLUMN, axis=1).columns)}

✅ PREPROCESSING STEPS COMPLETED:
   1. ✓ Loaded dataset using pandas
   2. ✓ Displayed dataset structure and summary
   3. ✓ Cleaned missing values
   4. ✓ Converted loan_status into binary target (0/1)
   5. ✓ Removed irrelevant columns
   6. ✓ Encoded categorical variables
   7. ✓ Applied feature scaling (StandardScaler)
   8. ✓ Handled class imbalance using SMOTE

📈 CLASS BALANCE:
   • Before SMOTE: {(y == 1).sum():,} defaults, {(y == 0).sum():,} fully paid ({(y == 1).sum() / (y == 0).sum():.4f} ratio)
   • After SMOTE: Both classes balanced at {(y_smote == 1).sum():,} samples each

🗂️ OUTPUT FILES:
   • preprocessed_data.csv    → Full processed dataset (balanced)
   • train_data.csv          → Training set ({train_df.shape[0]:,} samples, {train_df.shape[1]} features)
   • test_data.csv           → Test set ({test_df.shape[0]:,} samples, {test_df.shape[1]} features)

🎯 NEXT STEPS:
   → Phase 2: Exploratory Data Analysis (EDA)
   → Phase 3: Feature Engineering
   → Phase 4: Model Development & Training
   → Phase 5: Model Evaluation & Validation
   → Phase 6: Deployment & Insights

╔════════════════════════════════════════════════════════════════════════════╗
║  Data is now ready for model training and predictive analytics!          ║
╚════════════════════════════════════════════════════════════════════════════╝
""")